# Projet RTE — à quelle heure l'électricité est-elle la plus verte ?

Sources :
- éCO2mix national et régional (RTE, via ODRÉ) — données définitives et consolidées
- Données climatologiques horaires (Météo-France, via meteo.data.gouv.fr) — une station de référence par région

In [ ]:
%pip install requests

In [ ]:
import os
import requests
import pandas as pd

## 1. Téléchargement

In [ ]:
def telecharger(url, fichier):
    """Télécharge le fichier une seule fois."""
    if not os.path.exists(fichier):
        print("Téléchargement de", fichier, "...")
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(fichier, "wb") as f:
                for bloc in r.iter_content(chunk_size=1024*1024):
                    f.write(bloc)
    print("OK :", fichier)

odre = "https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/"
meteo_url = "https://meteofrance.s3.sbg.io.cloud.ovh.net/data/synchro_ftp/BASE/HOR/"

telecharger(odre + "eco2mix-national-cons-def/exports/csv", "eco2mix_national.csv")
telecharger(odre + "eco2mix-regional-cons-def/exports/csv", "eco2mix_regional.csv")

Une station météo de référence par région. Lille (pas de rayonnement solaire) et Rouen (rayonnement incomplet)
sont remplacées par Valenciennes et Caen.

In [ ]:
stations = pd.DataFrame([
    (11, "75", 75114001, "PARIS-MONTSOURIS"),
    (24, "45", 45055001, "ORLEANS"),
    (27, "21", 21473001, "DIJON-LONGVIC"),
    (28, "14", 14137001, "CAEN-CARPIQUET"),
    (32, "59", 59606004, "VALENCIENNES"),
    (44, "67", 67124001, "STRASBOURG-ENTZHEIM"),
    (52, "44", 44020001, "NANTES-BOUGUENAIS"),
    (53, "35", 35281001, "RENNES-ST JACQUES"),
    (75, "33", 33281001, "BORDEAUX-MERIGNAC"),
    (76, "31", 31069001, "TOULOUSE-BLAGNAC"),
    (84, "69", 69029001, "LYON-BRON"),
    (93, "13", 13054001, "MARIGNANE"),
], columns=["code_insee_region", "departement", "NUM_POSTE", "station"])

periodes = {"2020-2024": "previous-2020-2024", "2025-2026": "latest-2025-2026"}

for dep in stations["departement"]:
    for nom_court, nom_url in periodes.items():
        telecharger(meteo_url + f"H_{dep}_{nom_url}.csv.gz", f"meteo_{dep}_{nom_court}.csv.gz")

## 2. Chargement

In [ ]:
national = pd.read_csv("eco2mix_national.csv", sep=";", low_memory=False)
regional = pd.read_csv("eco2mix_regional.csv", sep=";", low_memory=False)

morceaux = []
for _, s in stations.iterrows():
    for nom_court in periodes:
        m = pd.read_csv(f"meteo_{s['departement']}_{nom_court}.csv.gz", sep=";",
                        usecols=["NUM_POSTE", "AAAAMMJJHH", "T", "FF", "RR1", "GLO"])
        m = m[m["NUM_POSTE"] == s["NUM_POSTE"]]          # on garde seulement la station choisie
        m["code_insee_region"] = s["code_insee_region"]   # on lui colle le code de sa région
        morceaux.append(m)
meteo = pd.concat(morceaux, ignore_index=True)

for nom, t in [("national", national), ("regional", regional), ("meteo", meteo)]:
    print(nom, t.shape)

## 3. Nettoyage

- **National** : les lignes à :15 et :45 sont vides (données au pas de 30 min) → supprimées.
  `prevision_j` est retirée (connue le jour même seulement : fuite de données pour un modèle J+1).
- **Régional** : `eolien` est lu comme du texte → converti en nombre ; colonnes batteries, `tco_*`, `tch_*` et `column_30` retirées (vides à plus de 50 %).
- **Météo** : date `AAAAMMJJHH` (UTC) convertie, colonnes renommées.
- On garde la période commune : depuis le 1er janvier 2020.

In [ ]:
garder_nat = ["date_heure", "consommation", "prevision_j1",
              "fioul", "charbon", "gaz", "nucleaire", "eolien", "solaire",
              "hydraulique", "pompage", "bioenergies", "ech_physiques", "taux_co2"]

national = national[garder_nat].copy()
national = national.dropna(subset=["consommation"])          # supprime les lignes :15 et :45 vides
national["date_heure"] = pd.to_datetime(national["date_heure"], utc=True)
national = national[national["date_heure"] >= "2020-01-01"]

print(national.shape)
national.isna().sum()

In [ ]:
garder_reg = ["code_insee_region", "libelle_region", "date_heure", "consommation",
              "thermique", "nucleaire", "eolien", "solaire", "hydraulique",
              "pompage", "bioenergies", "ech_physiques"]

regional = regional[garder_reg].copy()
regional["eolien"] = pd.to_numeric(regional["eolien"], errors="coerce")   # texte → nombre
regional[["nucleaire", "pompage"]] = regional[["nucleaire", "pompage"]].fillna(0)   # régions sans centrale : vide = 0
regional = regional.dropna(subset=["consommation"])
regional["date_heure"] = pd.to_datetime(regional["date_heure"], utc=True)
regional = regional[regional["date_heure"] >= "2020-01-01"]

print(regional.shape)
regional.isna().sum()

In [ ]:
meteo["date_heure"] = pd.to_datetime(meteo["AAAAMMJJHH"].astype(str), format="%Y%m%d%H", utc=True)
meteo = meteo[["code_insee_region", "date_heure", "T", "FF", "RR1", "GLO"]]
meteo = meteo.rename(columns={"T": "temperature", "FF": "vent", "RR1": "pluie", "GLO": "rayonnement"})

print(meteo.shape)
print(meteo.duplicated(subset=["code_insee_region", "date_heure"]).sum(), "doublons")
meteo.isna().sum()

## 4. Passage au pas horaire

éCO2mix est au pas de 30 min, la météo à l'heure : on fait la moyenne des deux demi-heures de chaque heure
(les valeurs sont des puissances en MW, la moyenne est donc la bonne agrégation).

In [ ]:
national["date_heure"] = national["date_heure"].dt.floor("h")
national_h = national.groupby("date_heure").mean().reset_index()

regional["date_heure"] = regional["date_heure"].dt.floor("h")
regional_h = (regional.groupby(["code_insee_region", "libelle_region", "date_heure"])
                      .mean()
                      .reset_index())

print("national_h", national_h.shape)
print("regional_h", regional_h.shape)

## 5. Jointure

- régional + météo sur **(code_insee_region, date_heure)** : chaque région reçoit la météo de sa station
- + national sur **date_heure** : le taux de CO₂ n'existe qu'au niveau France, chaque région reçoit celui de la même heure

`how="left"` : on garde toutes les heures d'énergie, même si la météo manque (les trous restent visibles).

In [ ]:
data = regional_h.merge(meteo, on=["code_insee_region", "date_heure"], how="left")

data = data.merge(national_h[["date_heure", "taux_co2", "consommation"]]
                      .rename(columns={"consommation": "consommation_france"}),
                  on="date_heure", how="left")

print("avant :", regional_h.shape, "→ après :", data.shape)   # le nombre de lignes ne doit pas changer
data.head()

## 6. Vérifications

In [ ]:
print("Période :", data["date_heure"].min(), "→", data["date_heure"].max())
print("Doublons (région, heure) :", data.duplicated(subset=["code_insee_region", "date_heure"]).sum())
print()
print("% de valeurs vides par colonne :")
print((data.isna().mean() * 100).round(2))

In [ ]:
# Nombre d'heures et part de météo manquante par région
data.groupby("libelle_region").agg(
    heures=("date_heure", "count"),
    debut=("date_heure", "min"),
    fin=("date_heure", "max"),
    meteo_manquante_pct=("temperature", lambda s: round(s.isna().mean() * 100, 2)),
)

**À noter** : Auvergne-Rhône-Alpes n'a pas de données RTE de mi-février à décembre 2025 dans le fichier source
(environ 7 600 heures). Ce n'est pas une erreur de jointure. À mentionner dans les limites du README.

## 7. Sauvegarde

In [ ]:
data.to_csv("data_regions_meteo_horaire.csv", index=False, sep=";")
national_h.to_csv("data_national_horaire.csv", index=False, sep=";")
print("Fichiers enregistrés")